<a href="https://colab.research.google.com/github/Sai-charan822/DSA03-NLP/blob/main/LAB/EXP20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 20. Simple TF-IDF Information Retrieval (IR) System
We will implement a TF-IDF vectorizer and calculation of cosine similarity from scratch to rank a collection of text documents relative to a user search query.

In [10]:
import math
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Corpus (Collection of Documents)
documents = {
    "Doc 1": "The quick brown fox jumps over the lazy dog.",
    "Doc 2": "The golden retriever puppy is very playful and friendly.",
    "Doc 3": "Python is a powerful programming language for data science and machine learning.",
    "Doc 4": "Large language models are trained on massive text databases.",
    "Doc 5": "Natural language processing applications include search engines and machine translation."
}

# Helper to preprocess text (lowercasing and simple alphanumeric tokenization)
def preprocess(text):
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalnum()]

# 1. Compute Term Frequency (TF) for a document: count(t, d) / total_words(d)
def compute_tf(doc_tokens):
    tf_dict = {}
    token_counts = Counter(doc_tokens)
    total_tokens = len(doc_tokens)
    for token, count in token_counts.items():
        tf_dict[token] = count / total_tokens
    return tf_dict

# 2. Compute Inverse Document Frequency (IDF) over the whole corpus
def compute_idf(all_docs_tokens):
    n_docs = len(all_docs_tokens)
    idf_dict = {}

    # Count how many documents contain each word
    all_unique_words = set(word for doc in all_docs_tokens for word in doc)
    for word in all_unique_words:
        doc_containing = sum(1 for doc in all_docs_tokens if word in doc)
        # IDF formula with smoothing to avoid division by zero
        idf_dict[word] = math.log((1 + n_docs) / (1 + doc_containing)) + 1
    return idf_dict

# 3. Compute Cosine Similarity between two vectors
def cosine_similarity(v1, v2):
    # v1 and v2 are dicts mapping word -> tf-idf score
    all_words = set(v1.keys()).union(set(v2.keys()))
    dot_product = sum(v1.get(word, 0) * v2.get(word, 0) for word in all_words)

    magnitude_1 = math.sqrt(sum(val ** 2 for val in v1.values()))
    magnitude_2 = math.sqrt(sum(val ** 2 for val in v2.values()))

    if magnitude_1 == 0 or magnitude_2 == 0:
        return 0.0
    return dot_product / (magnitude_1 * magnitude_2)

# Preprocess corpus
preprocessed_docs = {name: preprocess(content) for name, content in documents.items()}
all_tokens_list = list(preprocessed_docs.values())

# Compute TF and IDF globally
idfs = compute_idf(all_tokens_list)
doc_tfs = {name: compute_tf(tokens) for name, tokens in preprocessed_docs.items()}

# Compute TF-IDF vectors for documents
doc_tfidfs = {}
for name, tfs in doc_tfs.items():
    doc_tfidfs[name] = {word: tf_val * idfs[word] for word, tf_val in tfs.items()}

# Function to search the index with a query
def search(query, top_n=3):
    query_tokens = preprocess(query)
    query_tf = compute_tf(query_tokens)
    query_tfidf = {word: tf_val * idfs.get(word, 0) for word, tf_val in query_tf.items() if word in idfs}

    results = []
    for doc_name, doc_vector in doc_tfidfs.items():
        sim = cosine_similarity(query_tfidf, doc_vector)
        results.append((doc_name, sim))

    # Sort by similarity score descending
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_n]

# Test queries
test_queries = [
    "machine learning for language processing",
    "playful golden puppy"
]

for query in test_queries:
    print(f"Query: \"{query}\"")
    print("Ranked Results:")
    rankings = search(query)
    for rank, (doc_name, score) in enumerate(rankings, 1):
        print(f"  {rank}. {doc_name} (Similarity: {score:.4f}) -> \"{documents[doc_name]}\"")
    print("-" * 80)

Query: "machine learning for language processing"
Ranked Results:
  1. Doc 3 (Similarity: 0.4793) -> "Python is a powerful programming language for data science and machine learning."
  2. Doc 5 (Similarity: 0.3547) -> "Natural language processing applications include search engines and machine translation."
  3. Doc 4 (Similarity: 0.0762) -> "Large language models are trained on massive text databases."
--------------------------------------------------------------------------------
Query: "playful golden puppy"
Ranked Results:
  1. Doc 2 (Similarity: 0.6222) -> "The golden retriever puppy is very playful and friendly."
  2. Doc 1 (Similarity: 0.0000) -> "The quick brown fox jumps over the lazy dog."
  3. Doc 3 (Similarity: 0.0000) -> "Python is a powerful programming language for data science and machine learning."
--------------------------------------------------------------------------------
